# BigQuery Usage  
In this Notebook we'll upload data from a Pandas DataFrame to BigQuery and download the same DataFrame of data.

In [1]:
import pandas as pd

from google.cloud import bigquery
from pandas_gbq import schema

from pygcp import credentials, bq

In [2]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.max_colwidth = 1000

In [3]:
%store -r cred_file
%store -r project_id

In [4]:
gcp_credentials, project_id = credentials.create_credentials(cred_file=cred_file, 
                                                             project_id=project_id)

In [5]:
dataset_id  = 'test'
table_id    = 'loopback'
location_id = 'us'

In [6]:
bq.create_dataset?

Signature:
bq.create_dataset(
    gcp_credentials,
    project_id,
    dataset_id,
    location_id='us',
    exists_ok=False,
    verbose=False,
)
Docstring:
Create a dataset on in the specified project

See:
https://cloud.google.com/bigquery/docs/reference/standard-sql/data-definition-language#create_table_statement
https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.client.Client.html#google.cloud.bigquery.client.Client.create_dataset
https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.job.CopyJob.html#google.cloud.bigquery.job.CopyJob
for details

Parameters
----------
gcp_credentials :
    google.auth.compute_engine.credentials.Credentials
        OR
    google.oauth2.service_account.Credentials
project_id : str, GCP project to run the query in
dataset_id : str, name of dataset to create
location_id : str, GCP region to create dataset in. Default is 'us'.
exists_ok : bool, True = ignore “already exists” errors when creating the da

In [8]:
bq.create_dataset(gcp_credentials=gcp_credentials,
                  project_id=project_id,
                  dataset_id=dataset_id,
                  exists_ok=True)

Dataset(DatasetReference('sada-colin-dietrich', 'test'))

In [9]:
df_upload = pd.DataFrame({'a':[1,2,3], 
                          'b':[4.1,5.2,6.3], 
                          'c':['twelve', 'eleven', 'ten'],
                          'd':[['g', 'h'], ['i', 'j'], ['k', 'l']]
                         })

In [10]:
df_upload

,a,b,c,d
0,1,4.1,twelve,"[g, h]"
1,2,5.2,eleven,"[i, j]"
2,3,6.3,ten,"[k, l]"


In [11]:
schema.generate_bq_schema?

Signature: schema.generate_bq_schema(dataframe, default_type='STRING')
Docstring:
Given a passed dataframe, generate the associated Google BigQuery schema.

Arguments:
    dataframe (pandas.DataFrame): D
default_type : string
    The default big query type in case the type of the column
    does not exist in the schema.
File:      ~/miniconda3/envs/ds310a/lib/python3.10/site-packages/pandas_gbq/schema.py
Type:      function

In [12]:
pandas_gbq_schema = schema.generate_bq_schema(df_upload)
pandas_gbq_schema

{'fields': [{'name': 'a', 'type': 'INTEGER'},
  {'name': 'b', 'type': 'FLOAT'},
  {'name': 'c', 'type': 'STRING'},
  {'name': 'd', 'type': 'STRING'}]}

Column `d` has been auto detected as a string when it's a list strings. BigQuery will see this as a repeated value so we must specify the mode as `REPEATED`

In [13]:
pandas_gbq_schema['fields'][3]['mode'] = 'REPEATED'

Alternatively, we could set the schema directly with our own dictionary:  
```
pandas_gbq_schema_manual = {
'fields': [{'name': 'a', 'type': 'INTEGER'},
           {'name': 'b', 'type': 'FLOAT'},
           {'name': 'c', 'type': 'STRING'},
           {'name': 'd', 'type': 'STRING' 'mode': 'REPEATED'}
          ]
    }
```

In [14]:
pandas_gbq_schema

{'fields': [{'name': 'a', 'type': 'INTEGER'},
  {'name': 'b', 'type': 'FLOAT'},
  {'name': 'c', 'type': 'STRING'},
  {'name': 'd', 'type': 'STRING', 'mode': 'REPEATED'}]}

In [15]:
schema.to_google_cloud_bigquery?

Signature: schema.to_google_cloud_bigquery(pandas_gbq_schema)
Docstring:
Given a schema in pandas-gbq API format,
return a sequence of :class:`google.cloud.bigquery.schema.SchemaField`.
File:      ~/miniconda3/envs/ds310a/lib/python3.10/site-packages/pandas_gbq/schema.py
Type:      function

In [16]:
bq_schema = schema.to_google_cloud_bigquery(pandas_gbq_schema)
bq_schema

[SchemaField('a', 'INTEGER', 'NULLABLE', None, None, (), None),
 SchemaField('b', 'FLOAT', 'NULLABLE', None, None, (), None),
 SchemaField('c', 'STRING', 'NULLABLE', None, None, (), None),
 SchemaField('d', 'STRING', 'REPEATED', None, None, (), None)]

In [17]:
bq.job.load_config?

Signature: bq.job.load_config(**kwargs)
Docstring:
Set common BigQuery LoadJobConfig attributes

Notes
-----
1. Google Cloud Client API Documentation:
https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.job.LoadJob.html
https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.job.LoadJobConfig.html
https://cloud.google.com/bigquery/docs/reference/rest/v2/Job#jobconfigurationload

2. Not all attributes supported by the Python API and BigQuery API are represented in this wrapper,
see links for full list.


Keyword Arguments
-----------------
allow_jagged_rows : bool, optional
    Accept rows that are missing trailing optional columns. The missing values are treated as nulls. If false, records with missing trailing columns are treated as bad records, and if there are too many bad records, an invalid error is returned in the job result. The default value is false. Only applicable to CSV, ignored for other formats.

allow_quoted_newlines : bool

In [18]:
ljc = bq.job.load_config(write_disposition='WRITE_TRUNCATE',
                         create_disposition='CREATE_IF_NEEDED',
                         schema=bq_schema)
ljc

In [24]:
upload_kwargs = {
    'gcp_credentials': gcp_credentials,
    'df_upload': df_upload,
    'load_job_config': ljc,
    'project_id': project_id,
    'dataset_id': dataset_id,
    'table_id': table_id,
    'verbose': True
    }

In [25]:
bq.upload_dataframe?

Signature:
bq.upload_dataframe(
    gcp_credentials,
    df_upload,
    load_job_config,
    project_id,
    dataset_id,
    table_id,
    timeout=None,
    verbose=False,
)
Docstring:
Upload a dataframe to BigQuery

Notes
-----
Best to set `schema`, `create_disposition and `write_disposition`
attributes of the LoadJobConfig instance to avoid inferred errors
or data overwriting. See bq.job.load_config.

Parameters
----------
gcp_credentials :
    google.auth.compute_engine.credentials.Credentials
        OR
    google.oauth2.service_account.Credentials
df_upload : Pandas DataFrame, data to upload to BigQuery
load_job_config : class, google.cloud.bigquery.job.load.LoadJobConfig
    Load job configuration for this operation with
    matching bigquery.SchemaField attributes for all DataFrame columns
project_id : str, GCP project to create dataset in
dataset_id : str, BigQuery dataset to create table in
table_id : str, table name to upload data to
timeout : int, timeout in minutes. Optiona

In [26]:
load_metadata = bq.upload_dataframe(**upload_kwargs)

BigQuery query bytes: 0it [00:00, ?it/s]


totalBytesProcessed key missing
{'_polling': <google.api_core.retry.Retry object at 0x7f56ad7cb310>, '_result': None, '_exception': None, '_result_set': False, '_polling_thread': None, '_done_callbacks': [], '_properties': {'kind': 'bigquery#job', 'etag': 'NKLUsp4HRFqYufXmncuiAA==', 'id': 'sada-colin-dietrich:US.df9a612d-6c4a-46bc-8c31-595ee997a22d', 'selfLink': 'https://bigquery.googleapis.com/bigquery/v2/projects/sada-colin-dietrich/jobs/df9a612d-6c4a-46bc-8c31-595ee997a22d?location=US', 'user_email': 'colin-dev@sada-colin-dietrich.iam.gserviceaccount.com', 'configuration': {'load': {'schema': {'fields': [{'name': 'a', 'type': 'INTEGER', 'mode': 'NULLABLE'}, {'name': 'b', 'type': 'FLOAT', 'mode': 'NULLABLE'}, {'name': 'c', 'type': 'STRING', 'mode': 'NULLABLE'}, {'name': 'd', 'type': 'STRING', 'mode': 'REPEATED'}]}, 'destinationTable': {'projectId': 'sada-colin-dietrich', 'datasetId': 'test', 'tableId': 'loopback'}, 'createDisposition': 'CREATE_IF_NEEDED', 'writeDisposition': 'WRITE_T

In [27]:
load_metadata

{'kind': 'bigquery#job',
 'etag': 'HGVjbeF0tydo1VibDprbBw==',
 'id': 'sada-colin-dietrich:US.df9a612d-6c4a-46bc-8c31-595ee997a22d',
 'selfLink': 'https://bigquery.googleapis.com/bigquery/v2/projects/sada-colin-dietrich/jobs/df9a612d-6c4a-46bc-8c31-595ee997a22d?location=US',
 'user_email': 'colin-dev@sada-colin-dietrich.iam.gserviceaccount.com',
 'configuration': {'load': {'schema': {'fields': [{'name': 'a',
      'type': 'INTEGER',
      'mode': 'NULLABLE'},
     {'name': 'b', 'type': 'FLOAT', 'mode': 'NULLABLE'},
     {'name': 'c', 'type': 'STRING', 'mode': 'NULLABLE'},
     {'name': 'd', 'type': 'STRING', 'mode': 'REPEATED'}]},
   'destinationTable': {'projectId': 'sada-colin-dietrich',
    'datasetId': 'test',
    'tableId': 'loopback'},
   'createDisposition': 'CREATE_IF_NEEDED',
   'writeDisposition': 'WRITE_TRUNCATE',
   'sourceFormat': 'PARQUET',
   'parquetOptions': {'enableListInference': True}},
  'jobType': 'LOAD'},
 'jobReference': {'projectId': 'sada-colin-dietrich',
  'jo

In [28]:
query_text = f"""
SELECT * 
FROM `{project_id}.{dataset_id}.{table_id}`
"""

In [29]:
bq.job.query_config?

Signature: bq.job.query_config(**kwargs)
Docstring:
Set common BigQuery QueryJobConfig attributes

Notes
-----
1. Google Cloud Client API Documentation:
    https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.job.QueryJob.html

    https://googleapis.dev/python/bigquery/latest/generated/google.cloud.bigquery.job.QueryJobConfig.html

2. Attributes included are primarily for querying, some attributes
    for writing to new tables are excluded. See above links for details.

Keyword Arguments
-----------------
allow_large_results : bool, optional
    If true and query uses legacy SQL dialect, allows the query to produce arbitrarily large result tables at a slight cost in performance. Requires destinationTable to be set. For standard SQL queries, this flag is ignored and large results are always allowed. However, you must still set destinationTable when result size exceeds the allowed maximum response size.

clustering_fields : list of str, optional, defaults to No

In [30]:
qjc = bq.job.query_config()
qjc

In [35]:
download_kwargs = {
    'gcp_credentials': gcp_credentials,
    'project_id': project_id,
    'query_text': query_text,
    'query_job_config': qjc,
    'verbose': True
    }

In [36]:
bq.to_df?

Signature:
bq.to_df(
    gcp_credentials,
    project_id,
    query_job_config,
    query_text,
    timeout=None,
    verbose=False,
)
Docstring:
Upload a dataframe to BigQuery

Notes
-----

Parameters
----------
gcp_credentials : class instance of either
    google.auth.compute_engine.credentials.Credentials
        OR
    google.oauth2.service_account.Credentials
project_id : str, GCP project to run the query in
query_job_config : class instance of
    google.cloud.bigquery.job.load.QueryJobConfig
    Query job configuration for this operation
query_text : query to execute and convert to a DataFrame
timeout : int, Timeout in minutes. Optional, defaults to None
verbose : bool, print debug statements

Returns
-------
dict, state of google.cloud.bigquery.job.load.QueryJob._properties
    at completion of query job
Pandas DataFrame, query output
File:      ~/code/pygcp/pygcp/bq/base.py
Type:      function

In [37]:
metadata_query, df_download = bq.to_df(**download_kwargs)

BigQuery query bytes: 100%|██████████| 87/87 [00:00<00:00, 65080.16it/s]


In [38]:
metadata_query

{'kind': 'bigquery#job',
 'etag': 'H28uvV2ezPPdTl+qFT3TTg==',
 'id': 'sada-colin-dietrich:US.e05f5a91-a294-4292-9dff-5499cc9bf38c',
 'selfLink': 'https://bigquery.googleapis.com/bigquery/v2/projects/sada-colin-dietrich/jobs/e05f5a91-a294-4292-9dff-5499cc9bf38c?location=US',
 'user_email': 'colin-dev@sada-colin-dietrich.iam.gserviceaccount.com',
 'configuration': {'query': {'query': '\nSELECT * \nFROM `sada-colin-dietrich.test.loopback`\n',
   'destinationTable': {'projectId': 'sada-colin-dietrich',
    'datasetId': '_18baf94afd77c50a58159ed04b3d7b268d9e87aa',
    'tableId': 'anon3cdb1d130a946bc6b30f4abb5696e640ab4d890477beb8ab83e639894e65fe5a'},
   'writeDisposition': 'WRITE_TRUNCATE',
   'priority': 'INTERACTIVE',
   'useLegacySql': False},
  'jobType': 'QUERY'},
 'jobReference': {'projectId': 'sada-colin-dietrich',
  'jobId': 'e05f5a91-a294-4292-9dff-5499cc9bf38c',
  'location': 'US'},
 'statistics': {'creationTime': 1701475797798.0,
  'startTime': 1701475797913.0,
  'endTime': 17014

## Compare Upload and Download DataFrames  

In [39]:
df_upload.equals(df_download)

False

Column `a` is integers; by default the BigQuery API returns `INTEGER` as the Pandas type `Int64` instead of original `int64`

In [40]:
df_download.a = df_download.a.astype('int64')

In [41]:
df_upload.equals(df_download)

False

Column `d` again is a list of strings. The Pandas equality evaluations don't handle objects inside series well. It's easiest to convert the array or list to a JSON string for comparison. BigQuery returned the list as a Numpy `ndarry` so converting back to a list first is required for the JSON conversion.

In [42]:
import json

In [43]:
df_upload.d = df_upload.d.apply(json.dumps)
df_download.d = df_download.d.apply(lambda r: json.dumps(list(r)))

In [44]:
df_upload.equals(df_download)

True